# Python reference 8: files, paths and formats

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-08-files-paths-and-formats.ipynb)

`pathlib` for local paths and object-store keys, `open()` modes and streaming reads, atomic writes, `csv`, `json` and JSON Lines, compressed files, configuration sources, and `sqlite3` with parameters, transactions and upserts. Practise it in [Python practice 8: files, paths and formats](../practice/python-practice-08-files-paths-and-formats.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [pathlib](#pathlib) | [Building and taking apart paths](#building-and-taking-apart-paths), [Filesystem operations](#filesystem-operations) |
| [Reading and writing files](#reading-and-writing-files) | [`open()`](#open), [Temporary files, `shutil` and atomic writes](#temporary-files-shutil-and-atomic-writes) |
| [Data formats](#data-formats) | [`csv`](#csv), [`json` and JSON Lines](#json-and-json-lines), [Compressed files: `gzip`, `zipfile`](#compressed-files-gzip-zipfile), [Configuration: `tomllib`, `configparser`, environment variables](#configuration-tomllib-configparser-environment-variables) |
| [sqlite3](#sqlite3) | [Querying SQLite from Python](#querying-sqlite-from-python) |

## pathlib

### Building and taking apart paths

`pathlib.Path` represents a filesystem path as an object; `/` joins parts. `PurePosixPath` does the same without
touching the filesystem, which suits object-store keys (`s3://bucket/...` prefixes) and paths for another OS.

| member | for `Path("/data/raw/orders.csv.gz")` |
| --- | --- |
| `.name` | `orders.csv.gz` |
| `.stem` | `orders.csv` (only the last suffix removed) |
| `.suffix` / `.suffixes` | `.gz` / `['.csv', '.gz']` |
| `.parent` / `.parents` | `/data/raw` / all ancestors |
| `.parts` | `('/', 'data', 'raw', 'orders.csv.gz')` |
| `.with_suffix(".parquet")` / `.with_name("x")` / `.with_stem("y")` | a changed copy |
| `.relative_to(base)` | the path below `base` (`ValueError` if not under it) |
| `.resolve()` | absolute, symlinks and `..` resolved |
| `.as_posix()` | the path with forward slashes, as a string |

In [ ]:
from pathlib import Path, PurePosixPath

p = Path("/data/raw/orders.csv.gz")
print(p.name, p.stem, p.suffix, p.suffixes, p.parent, p.parts)
print(p.with_suffix(".parquet"), p.with_name("users.json"), p.relative_to("/data"), Path("/data") / "curated" / "orders")
key = PurePosixPath("warehouse/orders") / "date=2026-11-27" / "part-0000.parquet"
print(key, key.parent.name, str(key).split("/")[1])

### Filesystem operations

| method | does |
| --- | --- |
| `exists()`, `is_file()`, `is_dir()` | tests |
| `mkdir(parents=True, exist_ok=True)` | create a directory and missing parents |
| `iterdir()` | direct children (unordered; sort for determinism) |
| `glob("*.csv")` / `rglob("*.csv")` / `glob("**/*.csv")` | pattern matches, recursive with `rglob` or `**` |
| `read_text(encoding=)` / `write_text(text, encoding=)` / `read_bytes()` / `write_bytes()` | whole-file I/O |
| `stat().st_size`, `stat().st_mtime` | size in bytes, modification time |
| `unlink(missing_ok=True)` / `rmdir()` | delete a file / an empty directory |
| `rename(target)` / `replace(target)` | move; `replace` overwrites the target atomically on the same filesystem |

**Notes:** always pass `encoding="utf-8"`: the default depends on the platform locale (on Windows it is often not
UTF-8). Python 3.15 makes UTF-8 the default, but explicit is still clearer.

In [ ]:
import tempfile

root = Path(tempfile.mkdtemp())
(root / "raw" / "2026").mkdir(parents=True, exist_ok=True)
for name, text in [("raw/2026/a.csv", "id\n1\n"), ("raw/2026/b.csv", "id\n2\n3\n"), ("raw/notes.txt", "hi")]:
    (root / name).write_text(text, encoding="utf-8")
print(sorted(p.relative_to(root).as_posix() for p in root.rglob("*.csv")))
print(sorted(p.name for p in (root / "raw").iterdir()), (root / "raw/2026/b.csv").stat().st_size, "bytes")
(root / "raw/notes.txt").unlink()
print((root / "raw/notes.txt").exists(), (root / "missing.txt").read_text(encoding="utf-8") if (root / "missing.txt").exists() else "missing")

## Reading and writing files

### `open()`

`open(file, mode="r", encoding=None, newline=None, errors=None)` returns a file object; use it in a `with` block so
it is closed even on errors.

| mode | meaning |
| --- | --- |
| `"r"` / `"w"` / `"a"` | read / write (truncates) / append, text |
| `"x"` | create, failing with `FileExistsError` if it exists |
| `"rb"`, `"wb"` | binary: bytes in and out, no encoding |
| `"r+"` | read and write without truncating |

| pattern | when |
| --- | --- |
| `for line in f:` | line by line, constant memory; lines keep their `\n` |
| `f.read()` | small files only |
| `while chunk := f.read(1 << 20):` | binary chunks (hashing, copying) |
| `f.seek(offset)`, `f.tell()` | random access (binary mode for byte offsets) |

**Notes:** `errors="replace"` keeps reading through bad bytes (marking them with U+FFFD) instead of raising
`UnicodeDecodeError`; log how many were replaced. Text mode translates `\r\n` to `\n` on read unless `newline=""`.

In [ ]:
import hashlib

path = root / "events.log"
with open(path, "w", encoding="utf-8") as f:
    f.write("INFO start\nERROR boom\n\nINFO done\n")
with open(path, encoding="utf-8") as f:
    errors = [line.rstrip("\n") for line in f if line.startswith("ERROR")]
print(errors)

digest = hashlib.sha256()
with open(path, "rb") as f:
    while chunk := f.read(8):
        digest.update(chunk)
print(digest.hexdigest()[:16])

(root / "latin1.txt").write_bytes("café".encode("latin-1"))
print((root / "latin1.txt").read_text(encoding="utf-8", errors="replace"))
try:
    open(path, "x")
except FileExistsError as e:
    print("FileExistsError:", e.strerror)

### Temporary files, `shutil` and atomic writes

| tool | does |
| --- | --- |
| `tempfile.TemporaryDirectory()` | a directory deleted when the `with` block ends |
| `tempfile.NamedTemporaryFile(dir=..., delete=False)` | a temporary file with a real path |
| `shutil.copy2(src, dst)` / `shutil.copytree` | copy a file with metadata / a tree |
| `shutil.move(src, dst)` | move, across filesystems if needed |
| `shutil.rmtree(path)` | delete a directory tree (irreversible) |
| `os.replace(src, dst)` | rename, overwriting `dst` atomically on the same filesystem |

**Atomic write:** write to a temporary file **in the target's directory**, flush, then `os.replace` it over the target.
Readers then see either the old file or the new one, never a half-written file, even if the job crashes.

In [ ]:
import os
import shutil

def write_atomic(target, text):
    target = Path(target)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=target.parent, delete=False, suffix=".tmp") as tmp:
        tmp.write(text)
        tmp.flush()
        os.fsync(tmp.fileno())
    os.replace(tmp.name, target)

state = root / "state.json"
write_atomic(state, '{"watermark": "2026-11-26"}')
write_atomic(state, '{"watermark": "2026-11-27"}')
print(state.read_text(encoding="utf-8"), [p.name for p in root.glob("*.tmp")])

with tempfile.TemporaryDirectory() as scratch:
    shutil.copy2(state, Path(scratch) / "backup.json")
    print(sorted(os.listdir(scratch)))
print("scratch deleted:", not Path(scratch).exists())

## Data formats

### `csv`

| object | reads / writes |
| --- | --- |
| `csv.reader(f, delimiter=",", quotechar='"')` | lists of strings |
| `csv.DictReader(f, fieldnames=None)` | dicts keyed by the header row (or `fieldnames`) |
| `csv.writer(f, quoting=csv.QUOTE_MINIMAL, lineterminator="\r\n")` | `writerow(list)`, `writerows` |
| `csv.DictWriter(f, fieldnames, extrasaction="raise")` | `writeheader()`, `writerow(dict)` |

**Notes:** open CSV files with `newline=""` so the module handles quoted newlines and line endings itself. Every
value comes back as a **string** (empty fields are `""`), so convert types explicitly. `csv` handles quoting of
commas, quotes and newlines; never build CSV with `",".join`.

In [ ]:
import csv
import io

text = 'id,name,amount\n1,"Smith, J",120.5\n2,"say ""hi""",\n'
rows = list(csv.DictReader(io.StringIO(text)))
print(rows)
typed = [{"id": int(r["id"]), "name": r["name"], "amount": float(r["amount"]) if r["amount"] else None} for r in rows]
print(typed)

out = io.StringIO()
writer = csv.DictWriter(out, fieldnames=["id", "name", "amount"], lineterminator="\n")
writer.writeheader()
writer.writerows(typed)
print(out.getvalue())

### `json` and JSON Lines

| function | parameters worth knowing |
| --- | --- |
| `json.dumps(obj)` / `json.dump(obj, f)` | `indent=2`, `sort_keys=True`, `separators=(",", ":")` (compact), `ensure_ascii=False` (keep non-ASCII), `default=func` (convert unsupported types) |
| `json.loads(s)` / `json.load(f)` | `parse_float=Decimal` (exact money), `object_hook=func` (convert each dict) |

JSON has no date, Decimal, set or tuple types: tuples become lists, and the rest need `default=`. **JSON Lines**
(`.jsonl`, one JSON document per line) is the usual format for event streams and logs because it can be appended to
and read line by line.

In [ ]:
import json
from datetime import datetime
from decimal import Decimal

def encode(value):
    if isinstance(value, Decimal):
        return str(value)
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"not JSON serialisable: {type(value).__name__}")

record = {"id": 7, "amount": Decimal("310.10"), "at": datetime(2026, 11, 27, 9), "tags": {"b", "a"}, "city": "Zürich", "pair": (1, 2)}
print(json.dumps(record, default=encode, sort_keys=True, ensure_ascii=False, separators=(",", ":")))
print(json.loads('{"amount": 0.1}', parse_float=Decimal))

jsonl = "\n".join(json.dumps(r) for r in [{"id": 1}, {"id": 2}]) + "\n"
print([json.loads(line) for line in jsonl.splitlines() if line.strip()])

### Compressed files: `gzip`, `zipfile`

`gzip.open(path, "rt", encoding="utf-8")` reads a `.gz` file as text, line by line, exactly like `open`; `"wt"`
writes. `bz2.open` and `lzma.open` work the same way. `zipfile.ZipFile` reads and writes multi-file archives:
`namelist()`, `open(name)`, `writestr(name, data)`.

In [ ]:
import gzip
import zipfile

gz = root / "events.jsonl.gz"
with gzip.open(gz, "wt", encoding="utf-8") as f:
    for i in range(1000):
        f.write(json.dumps({"id": i, "status": "paid"}) + "\n")
with gzip.open(gz, "rt", encoding="utf-8") as f:
    print(sum(1 for _ in f), "lines,", gz.stat().st_size, "bytes compressed,", gz.read_bytes()[:2])

archive = root / "export.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.writestr("orders.csv", "id\n1\n")
    z.writestr("users.csv", "id\n9\n")
with zipfile.ZipFile(archive) as z:
    print(z.namelist(), z.read("users.csv").decode())

### Configuration: `tomllib`, `configparser`, environment variables

| source | read with | notes |
| --- | --- | --- |
| TOML (`pyproject.toml`, app config) | `tomllib.load(f)` (binary file) / `tomllib.loads(text)` | read-only in the standard library (3.11+) |
| INI files | `configparser.ConfigParser()` | values are strings; `getint`, `getboolean` convert |
| environment | `os.environ.get("NAME", default)` | strings only; parse numbers and booleans yourself |

In [ ]:
import tomllib

config = tomllib.loads("""
[job]
name = "daily_orders"
retries = 3
start = 2026-11-27

[source]
paths = ["s3://raw/orders", "s3://raw/refunds"]
""")
print(config["job"], config["source"]["paths"])
os.environ["JOB_RETRIES"] = "5"
print(int(os.environ.get("JOB_RETRIES", config["job"]["retries"])), os.environ.get("JOB_DEBUG", "false").lower() in {"1", "true", "yes"})

## sqlite3

### Querying SQLite from Python

`sqlite3` ships with Python: a full SQL database in one file (or `":memory:"`), useful for tests, local staging and
small lookups.

| call | does |
| --- | --- |
| `sqlite3.connect(path)` | open a connection |
| `conn.execute(sql, params)` | run one statement; `?` placeholders (or `:name` with a dict) |
| `conn.executemany(sql, seq_of_params)` | bulk insert |
| `cursor.fetchone()` / `fetchall()` / iterate | read results as tuples |
| `conn.row_factory = sqlite3.Row` | rows addressable by column name |
| `with conn:` | a transaction: commits on success, rolls back on an exception (it does **not** close) |

**Notes:** always pass values as parameters, never with f-strings: parameters prevent SQL injection and handle quoting.
SQLite supports `INSERT ... ON CONFLICT(key) DO UPDATE` (upsert), window functions and CTEs.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, region TEXT, amount REAL)")
with conn:
    conn.executemany("INSERT INTO orders VALUES (?, ?, ?)", [(1, "north", 120.0), (2, "south", 70.0), (3, "north", 50.0)])
with conn:
    conn.execute(
        "INSERT INTO orders VALUES (:id, :region, :amount) ON CONFLICT(order_id) DO UPDATE SET amount = excluded.amount",
        {"id": 3, "region": "north", "amount": 55.0},
    )
rows = conn.execute("SELECT region, COUNT(*) AS n, SUM(amount) AS total FROM orders GROUP BY region ORDER BY region").fetchall()
print([dict(r) for r in rows])
try:
    with conn:
        conn.execute("INSERT INTO orders VALUES (4, 'east', 10)")
        conn.execute("INSERT INTO orders VALUES (1, 'dup', 0)")
except sqlite3.IntegrityError as e:
    print("IntegrityError:", e, "| order 4 rolled back:", conn.execute("SELECT COUNT(*) FROM orders WHERE order_id = 4").fetchone()[0])
user_input = "north' OR '1'='1"
print(conn.execute("SELECT COUNT(*) FROM orders WHERE region = ?", (user_input,)).fetchone()[0], "rows for the injection attempt")
conn.close()

**Next:** practise these entries in [Python practice 8: files, paths and formats](../practice/python-practice-08-files-paths-and-formats.ipynb).